In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
trn_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
tst_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')


In [ ]:
trn_df.head()


In [ ]:
trn_df['input'] = trn_df['text'].astype(str)
tst_df['input'] = tst_df['text'].astype(str)


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(trn_df[['input', 'target']].rename(columns={'target': 'labels'}))
test_ds = Dataset.from_pandas(tst_df[['id', 'input']])


In [ ]:
from transformers import AutoTokenizer

model_name = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['input'], truncation=True, padding='max_length')


In [ ]:
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)


In [ ]:
ds = train_ds.train_test_split(test_size=0.1, seed=42)
ds


In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

batch_size = 32
epochs = 1
lr = 8e-5

training_args = TrainingArguments(
    output_dir="outputs",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    num_train_epochs=epochs,
    do_eval=True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="logs",
    fp16=True,
    report_to="none"
)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds)}


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
preds = trainer.predict(test_ds)
test_preds = np.argmax(preds.predictions, axis=1)

In [ ]:
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
submission['target'] = test_preds
submission.to_csv('submission.csv', index=False)
submission.head()
